In [1]:
import os
import hashlib
import time
from pathlib import Path
import wandb
from tqdm.notebook import tqdm

# Opciones: ajusta según necesites
PROJECT = "imagenette"          # nombre del proyecto en W&B
ENTITY = None                   # si usas una entidad/team, poner "mi_entidad", sino None
MODELS_DIR = Path("5_final_models")  # carpeta con los .pth
WANDB_TEMP = Path(r"C:\wandb_temp")  # evita AppData bloqueado en Windows

# --- utilidades ---
def sha256_of_file(path: Path, chunk_size: int = 4 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def upload_models_as_artifacts(models_dir: Path = MODELS_DIR,
                               project: str = PROJECT,
                               entity: str | None = ENTITY,
                               wandb_run_name: str | None = None,
                               tag: str | None = None,
                               dry_run: bool = False):
    models_dir = Path(models_dir)
    if not models_dir.exists() or not models_dir.is_dir():
        raise FileNotFoundError(f"La carpeta {models_dir} no existe.")

    # forzar directorio temporal de wandb en Windows si hace falta
    os.environ.setdefault("WANDB_DATA_DIR", str(WANDB_TEMP))
    WANDB_TEMP.mkdir(parents=True, exist_ok=True)

    print("Iniciando sesión en W&B (si no estás logueado te pedirá el API key)...")
    wandb.login()  # si ya estás logueado usará la sesión

    run = wandb.init(project=project, entity=entity, name=wandb_run_name or "upload_models_artifacts", reinit=True, job_type="upload_models")
    try:
        pths = sorted(models_dir.glob("*.pth"))
        if not pths:
            print(f"No se encontraron archivos .pth en {models_dir}")
            return []

        uploaded = []
        print(f"Se encontraron {len(pths)} archivos .pth — comenzando subida...")
        for p in tqdm(pths, desc="Subiendo .pth"):
            fname = p.name
            stem = p.stem
            size_bytes = p.stat().st_size
            sha256 = sha256_of_file(p)

            metadata = {
                "filename": fname,
                "local_path": str(p.resolve()),
                "size_bytes": int(size_bytes),
                "size_mb": round(size_bytes / (1024 * 1024), 3),
                "sha256": sha256,
                "uploaded_at": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            }

            artifact_name = f"{stem}_model"
            print(f"\n→ Subiendo '{fname}' como artifact '{artifact_name}' ({metadata['size_mb']} MB)...")

            artifact = wandb.Artifact(name=artifact_name, type="model", metadata=metadata)
            artifact.add_file(str(p.resolve()))

            if dry_run:
                print("  [dry_run] no se sube realmente.")
                uploaded.append({"artifact_name": artifact_name, "metadata": metadata, "status": "dry_run"})
                continue

            logged = run.log_artifact(artifact)
            logged.wait()  # esperar a que termine la subida
            print(f"   ✅ Subida completada. Artifact id: {logged.id}")
            if tag:
                # añadir tag al artifact en la UI (metadatos ya llevan info)
                logged.aliases = [tag]

            uploaded.append({"artifact_name": artifact_name, "metadata": metadata, "wandb_id": logged.id})

        print("\nTodas las subidas finalizadas.")
        return uploaded
    finally:
        wandb.finish()


c:\Users\kidni\Desktop\imagenette\.venv\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kidni\Desktop\imagenette\.venv\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Ann

In [2]:
uploaded = upload_models_as_artifacts(models_dir=Path("5_final_models"), project="imagenette", entity="kidnixt-ort", tag="final_models_v1")
print(uploaded)

Iniciando sesión en W&B (si no estás logueado te pedirá el API key)...


wandb: Currently logged in as: kidnixt (kidnixt-ort) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Se encontraron 6 archivos .pth — comenzando subida...


Subiendo .pth:   0%|          | 0/6 [00:00<?, ?it/s]


→ Subiendo 'densenet_bigger.pth' como artifact 'densenet_bigger_model' (118.891 MB)...
   ✅ Subida completada. Artifact id: QXJ0aWZhY3Q6MjE2OTM1MzE4Ng==

→ Subiendo 'densenet_monster.pth' como artifact 'densenet_monster_model' (195.022 MB)...
   ✅ Subida completada. Artifact id: QXJ0aWZhY3Q6MjE2OTM1Mzg2Ng==

→ Subiendo 'densenet_standard.pth' como artifact 'densenet_standard_model' (37.967 MB)...
   ✅ Subida completada. Artifact id: QXJ0aWZhY3Q6MjE2OTM1NDcyNg==

→ Subiendo 'lenet_160_efficient.pth' como artifact 'lenet_160_efficient_model' (10.046 MB)...
   ✅ Subida completada. Artifact id: QXJ0aWZhY3Q6MjE2OTM1NDg4NA==

→ Subiendo 'lenet_160_oversized.pth' como artifact 'lenet_160_oversized_model' (42.077 MB)...
   ✅ Subida completada. Artifact id: QXJ0aWZhY3Q6MjE2OTM1NDk5MA==

→ Subiendo 'tiny_cnn_final.pth' como artifact 'tiny_cnn_final_model' (0.492 MB)...
   ✅ Subida completada. Artifact id: QXJ0aWZhY3Q6MjE2OTM1NTE5MQ==

Todas las subidas finalizadas.


[{'artifact_name': 'densenet_bigger_model', 'metadata': {'filename': 'densenet_bigger.pth', 'local_path': 'C:\\Users\\kidni\\Desktop\\imagenette\\5_final_models\\densenet_bigger.pth', 'size_bytes': 124666643, 'size_mb': 118.891, 'sha256': 'da87c6b711c90e173db766aa18851d31e5b3acce6668a7da3aa1c6f2a1644428', 'uploaded_at': '2025-10-26 12:50:40'}, 'wandb_id': 'QXJ0aWZhY3Q6MjE2OTM1MzE4Ng=='}, {'artifact_name': 'densenet_monster_model', 'metadata': {'filename': 'densenet_monster.pth', 'local_path': 'C:\\Users\\kidni\\Desktop\\imagenette\\5_final_models\\densenet_monster.pth', 'size_bytes': 204495457, 'size_mb': 195.022, 'sha256': '1d8eb4a6b18c1286f03b21a2f93f1dfcebc11bf70a5c5c5c5b1129aac8014fbd', 'uploaded_at': '2025-10-26 12:51:18'}, 'wandb_id': 'QXJ0aWZhY3Q6MjE2OTM1Mzg2Ng=='}, {'artifact_name': 'densenet_standard_model', 'metadata': {'filename': 'densenet_standard.pth', 'local_path': 'C:\\Users\\kidni\\Desktop\\imagenette\\5_final_models\\densenet_standard.pth', 'size_bytes': 39811499, 'si